# Smart Image Selection

Selects a representative subset from a large class folder: extract ResNet50 features, reduce with UMAP, cluster with KMeans, and sample evenly across clusters. Plots the feature space and copies the selected images.

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import umap
import shutil

class ImageFolderFlat(Dataset):
    def __init__(self, folder):
        self.paths = [
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.transform(img), self.paths[i]

def extract_features(folder, device, batch_size=64):
    backbone = models.resnet50(weights='IMAGENET1K_V1')
    backbone.fc = torch.nn.Identity()
    backbone = backbone.to(device).eval()
    
    dataset    = ImageFolderFlat(folder)
    loader     = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    all_features, all_paths = [], []
    
    print(f"  Extracting features from {len(dataset)} images...", end=' ')
    with torch.no_grad():
        for imgs, paths in loader:
            feats = backbone(imgs.to(device)).cpu().numpy()
            all_features.extend(feats)
            all_paths.extend(paths)
    print("Done!")
    
    return np.array(all_features), all_paths

def analyze_and_sample(folder, output_dir, n_select, n_clusters=10, 
                        dataset_name="Dataset", device='cuda'):
    os.makedirs(output_dir, exist_ok=True)
    
    features, paths = extract_features(folder, device)
    
    print(f"  Running UMAP on {len(features)} images...", end=' ')
    scaler   = StandardScaler()
    features_scaled = scaler.fit_transform(features)
    
    reducer  = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    coords_2d = reducer.fit_transform(features_scaled)
    print("Done!")
    
    print(f"  Clustering into {n_clusters} groups...", end=' ')
    kmeans  = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(features_scaled)
    print("Done!")
    
    samples_per_cluster = n_select // n_clusters
    remainder           = n_select  % n_clusters
    selected_paths      = []
    shortfall           = 0
    
    for c in range(n_clusters):
        cluster_indices = np.where(cluster_labels == c)[0]
        n_take = samples_per_cluster + (1 if c < remainder else 0) + shortfall
        n_take = min(n_take, len(cluster_indices))
        shortfall = (samples_per_cluster + (1 if c < remainder else 0)) - n_take
        chosen = np.random.choice(cluster_indices, n_take, replace=False)
        selected_paths.extend([paths[i] for i in chosen])
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(f'{dataset_name} — Feature Space Analysis', 
                 fontsize=14, fontweight='bold')
    
    colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))
    for c in range(n_clusters):
        mask = cluster_labels == c
        axes[0].scatter(coords_2d[mask, 0], coords_2d[mask, 1],
                       c=[colors[c]], label=f'Cluster {c} ({mask.sum()})',
                       alpha=0.6, s=15)
    axes[0].set_title(f'All {len(paths)} Images — Colored by Cluster')
    axes[0].legend(fontsize=7, markerscale=2)
    axes[0].set_xlabel('UMAP-1')
    axes[0].set_ylabel('UMAP-2')
    
    selected_set     = set(selected_paths)
    selected_mask    = np.array([p in selected_set for p in paths])
    
    axes[1].scatter(coords_2d[~selected_mask, 0], coords_2d[~selected_mask, 1],
                   c='lightgray', alpha=0.4, s=10, label='Not selected')
    axes[1].scatter(coords_2d[selected_mask, 0],  coords_2d[selected_mask, 1],
                   c='red', alpha=0.8, s=20, label=f'Selected ({n_select})')
    axes[1].set_title(f'Selected {n_select} Images — Spread Across Feature Space')
    axes[1].legend(fontsize=10, markerscale=2)
    axes[1].set_xlabel('UMAP-1')
    axes[1].set_ylabel('UMAP-2')
    
    plt.tight_layout()
    plot_path = os.path.join(output_dir, f"{dataset_name}_feature_space.png")
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plot saved to {plot_path}")
    
    selected_dir = os.path.join(output_dir, "selected")
    os.makedirs(selected_dir, exist_ok=True)
    for p in selected_paths:
        shutil.copy(p, os.path.join(selected_dir, os.path.basename(p)))
    
    print(f"{len(selected_paths)} images copied to {selected_dir}")
    print(f"\n  Cluster breakdown:")
    for c in range(n_clusters):
        total_in_cluster    = int(np.sum(cluster_labels == c))
        selected_in_cluster = sum(
            1 for i, p in enumerate(paths)
            if p in selected_set and cluster_labels[i] == c
        )
        print(f"    Cluster {c:2d}: {selected_in_cluster:3d} selected / {total_in_cluster:4d} total")
    
    return selected_paths, coords_2d, cluster_labels

## Per-dataset selection runs

Each cell below runs the pipeline on one source/class folder.

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

selected_roboflow, coords_2d, cluster_labels = analyze_and_sample(
    folder       = r"D:\Desktop\Sesame Datasets\Sesame_Preprocessing_Study\Trial_A (Resize)\RSFD_Clean\Bacterial Blight Infected",
    output_dir   = r"D:\Desktop\Sesame Datasets\Combining Datasets\RFSD300_160\Bacterial Blight Infected",
    n_select     = 300,
    n_clusters   = 10,
    dataset_name = "RFSD_Bacterial",
    device       = DEVICE
)

dataset = ImageFolderFlat(r"D:\Desktop\Sesame Datasets\Sesame_Preprocessing_Study\Trial_A (Resize)\RSFD_Clean\Bacterial Blight Infected")
paths = dataset.paths

In [ ]:
selected_powdery,coords_2d, cluster_labels = analyze_and_sample(
    folder       = r"D:\Desktop\Sesame Datasets\Sesame_Preprocessing_Study\Trial_A (Resize)\RFSD_Clean\Powdery Mildew",
    output_dir   = r"D:\Desktop\Sesame Datasets\Combining Datasets\RFSD300_160\Powdery Mildew",
    n_select     = 200,
    n_clusters   = 5,
    dataset_name = "RFSD_Powdery",
    device       = DEVICE
)

In [ ]:
selected_mendeley_h, coords_2d2, cluster_labels2 = analyze_and_sample(
    folder       = r"D:\Desktop\Sesame Datasets\Sesame_Preprocessing_Study\Trial_A (Resize)\MSLD_Clean\Healthy Leaf",
    output_dir   = r"D:\Desktop\Sesame Datasets\Combining Datasets\MSLD300_300\Healthy Leaf",
    n_select     = 300,
    n_clusters   = 10,
    dataset_name = "MSLD_Healthy",
    device       = DEVICE
)

dataset = ImageFolderFlat(r"D:\Desktop\Sesame Datasets\Sesame_Preprocessing_Study\Trial_A (Resize)\MSLD_Clean\Healthy Leaf")
paths   = dataset.paths

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("Top Blob vs Bottom Blob — What's Different?", 
             fontsize=14, fontweight='bold')

top_indices    = np.where(coords_2d2[:, 1] > -28)[0]
bottom_indices = np.where(coords_2d2[:, 1] < -28)[0]

top_sample    = np.random.choice(top_indices,    4, replace=False)
bottom_sample = np.random.choice(bottom_indices, 4, replace=False)

for i, idx in enumerate(top_sample):
    img = Image.open(paths[idx]).convert('RGB')
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"TOP BLOB\n{os.path.basename(paths[idx])[:25]}", fontsize=8)
    axes[0, i].axis('off')

for i, idx in enumerate(bottom_sample):
    img = Image.open(paths[idx]).convert('RGB')
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"BOTTOM BLOB\n{os.path.basename(paths[idx])[:25]}", fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig("mendeley_blob_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Top blob:    {len(top_indices)} images")
print(f"Bottom blob: {len(bottom_indices)} images")

In [ ]:
selected_mendeley_b, coords_2d, cluster_labels= analyze_and_sample(
    folder       = r"D:\Desktop\Sesame Datasets\Sesame_Preprocessing_Study\Trial_A (Resize)\MSLD_Clean\Bacteria Blight Infected",
    output_dir   = r"D:\Desktop\Sesame Datasets\Combining Datasets\MSLD300_300\Bacteria Blight Infected",
    n_select     = 300,
    n_clusters   = 8,
    dataset_name = "MSLD_Bacterial",
    device       = DEVICE
)

dataset = ImageFolderFlat(r"D:\Desktop\Sesame Datasets\Sesame_Preprocessing_Study\Trial_A (Resize)\MSLD_Clean\Bacteria Blight Infected")
paths   = dataset.paths